# 08 · Explorar datos con pandas

Los CSV están en [**PySpark/data**](../data/). Ejecuta las celdas de arriba abajo. Necesitas pandas, matplotlib, seaborn y SciPy en el kernel de Jupyter. Si faltan, puedes instalar las tres bibliotecas en una celda con %pip install pandas matplotlib seaborn scipy.

El **análisis exploratorio de datos (EDA)** empieza por entender qué representa cada fila y detectar qué debemos revisar. Después veremos tres preguntas sobre la forma de los salarios, la relación entre salario y kilómetros, y la asociación entre grupo profesional y tener coche.


## 1. Comprender el problema y los datos

**[empleados_curso.csv](../data/empleados_curso.csv)** contiene 1.007 fichas de una encuesta de empleados adaptada para clase. Una fila es una ficha y una persona puede aparecer varias veces. Los nombres y DNI se retiraron.

| Columna | Qué significa |
|---|---|
| id_persona | Código de la persona encuestada. |
| id_registro | Código de la ficha. Las copias de una ficha comparten este código. |
| ciudad | Ciudad indicada en la encuesta. |
| grupo | Grupo profesional. Las categorías del ejercicio van de Grup A a Grup D. |
| salario_original | Salario mensual en euros tal como se registró. |
| tiene_coche | Respuesta sobre si tiene coche. True es sí. False es no. |
| km_anuales | Kilómetros recorridos en un año. |

**[ciudades_contexto.csv](../data/ciudades_contexto.csv)** contiene 36 filas. Cada fila asocia una ciudad con el país en el que está y con un índice de coste de vida de la tabla del curso.

| Columna | Qué significa |
|---|---|
| ciudad | Nombre de la ciudad en el archivo de contexto. |
| pais_contexto | País al que corresponde esa ciudad. |
| indice_coste_vida | Índice de la ciudad. Se expresa en puntos. |

Podemos usar la ciudad para juntar las tablas. Antes hay que decidir qué hacer con nombres como Paris, que aparece asociado a dos países.

## 2. Cargar e inspeccionar el dataset

Mostramos cinco filas para orientarnos. Después miramos el tamaño y los tipos de las columnas. El salario aparece como texto porque mezcla cifras con símbolos y palabras.

In [ ]:
import pandas as pd

empleados = pd.read_csv('../data/empleados_curso.csv')
ciudades = pd.read_csv('../data/ciudades_contexto.csv')

display(empleados.head())
display(ciudades.head())
print('Empleados (filas, columnas):', empleados.shape)
print('Ciudades (filas, columnas):', ciudades.shape)
display(empleados.dtypes)
display(ciudades.dtypes)

In [ ]:
empleados = pd.read_csv('../data/empleados_curso.csv', encoding='utf-8')
empleados.head()

## 3. Evaluar la calidad de los datos

### Datos vacíos

Hay 349 fichas sin respuesta sobre el coche. Un kilometraje vacío puede corresponder a alguien sin coche o a alguien que tenía coche y dejó el campo en blanco. En la columna de grupo aparece Z, fuera de las categorías del ejercicio.

Vamos a apartar las fichas que no podemos atribuir a una persona única y a preparar las columnas necesarias para el análisis. Conservamos los CSV y el DataFrame original.

In [ ]:
display(empleados.isna().sum())
display(empleados['grupo'].value_counts(dropna=False))

### Distinguir copias de fichas y personas repetidas

Quitamos las copias por el código de ficha. Apartamos las fichas sin código de persona. Si aún quedan varias fichas distintas para una persona, apartamos todas las suyas porque desconocemos cuál está vigente.

Compara los cuatro recuentos. La diferencia entre dos recuentos consecutivos te dice cuántas fichas apartaste en ese paso.

In [ ]:
empleados.drop_duplicates(subset='id_registro')

In [ ]:
empleados['id_registro'].duplicated().sum()

In [ ]:
empleados[empleados['id_registro'].duplicated(keep=False)].sort_values(['id_registro', 'id_persona'])

In [ ]:
base = empleados.drop_duplicates()

print('Una ficha por persona:', len(base))

### Salario

En este CSV los importes escritos con cifras están en euros enteros. La coma y el punto separan los miles. Por ejemplo, «1,469 €» son 1469 euros. En otros archivos una coma o un punto puede marcar decimales.

Primero quitamos las marcas de moneda, los espacios y los separadores. Luego traducimos tres importes escritos en catalán. El resultado se guarda en una columna nueva. Las respuestas «n/d» y «molts» quedan sin importe.

In [ ]:
base['salario_original'].unique()

In [ ]:
texto = "â‚¬"
texto.encode("cp1252").decode("utf-8") 

In [ ]:
importe = base['salario_original'].str.lower()
importe.str.replace("â‚¬", "", regex=False)

In [ ]:
importes_escritos = {'mil vuit-cents': '1800', 'dos mil': '2000', 'tres mil': '3000'}


importe = importe.str.replace("â‚¬", "", regex=False)
importe = importe.replace(importes_escritos)
importe = importe.str.replace(",", "", regex=False)
importe = importe.str.replace('euros', '', regex=False)
importe


In [ ]:
base['salario_eur'] = pd.to_numeric(importe, errors='coerce')

print(base['salario_eur'].isnull().sum())
display(base[['salario_original', 'salario_eur']].head(8))
display(base.loc[base['salario_eur'].isna(), 'salario_original'].value_counts(dropna=False))

### Grupo y kilometraje

Dejamos Z sin categoría. Un kilometraje negativo es imposible y queda vacío. También dejamos vacío el kilometraje cuando la persona responde que no tiene coche. Si la respuesta sobre el coche falta, conservamos el kilometraje registrado y dejamos esa ficha fuera de la comparación entre personas que declararon tener coche.

In [ ]:
base['grupo'].value_counts(dropna=False)

In [ ]:
base['grupo'] = base['grupo'].replace('Z', pd.NA)
base

| `tiene_coche` | `km_anuales` | Cómo interpretarlo |
|---|---|---|
| `True` | >0 | Tiene coche y hay un kilometraje registrado. |
| `True` | `NaN` | Tiene coche, pero falta el kilometraje. |
| `False` | `NaN` | No tiene coche, así que el kilometraje no aplica. |
| `False` | Un número | Hay una incoherencia que merece revisión. En el ejercicio dejamos vacío el kilometraje limpio. |
| `NaN` | `NaN` | Faltan ambos datos. |
| `NaN` | Un número | Hay kilometraje registrado, pero desconocemos si tiene coche. Conservamos el dato y dejamos esa fila fuera del análisis de quienes declararon tener coche. |

¿Hay < 0?

In [ ]:
sin_coche = base['tiene_coche'].eq(False)
km_negativos = base['km_anuales'] < 0

print('Kilometrajes negativos:', km_negativos.sum())
print('Kilometraje registrado pese a responder que no tiene coche:',
      (sin_coche & base['km_anuales'].notna()).sum())

base['km_limpios'] = base['km_anuales'].mask(sin_coche | km_negativos)

## 4. Estadística descriptiva

Resumimos el salario por grupo. **Personas** cuenta todas las fichas del grupo y **con salario** cuenta las que tienen un importe. La **media** es el promedio, la **mediana** es el valor central al ordenar los importes y la **desviación típica** mide cuánto se dispersan alrededor de la media. Comparamos también los mínimos y los máximos.

In [ ]:
resumen = base.groupby('grupo')['salario_eur'].agg(
    personas='size', con_salario='count', media='mean', mediana='median',
    minimo='min', maximo='max', desviacion='std',
)
display(resumen.round(2))

## 5. Analizar distribuciones

### ¿Qué forma tienen los salarios? Normalidad con Shapiro–Wilk

Miramos los salarios del **grupo A** como ejemplo. El histograma agrupa los importes por intervalos y muestra cuántas personas hay en cada uno. Fíjate en si la distribución tiene una sola cima, es aproximadamente simétrica o tiene una cola más larga hacia un lado.

En Shapiro–Wilk, la **hipótesis nula (H₀)** dice que los salarios observados proceden de una población con distribución normal. En clase usaremos un umbral de **0,05**. Si el valor p es menor que 0,05, rechazamos H₀. Si es igual o mayor, la muestra aporta evidencia insuficiente para rechazarla. Lee el resultado junto con el histograma. Con muchas observaciones, el test puede detectar desviaciones pequeñas.

In [ ]:
from scipy import stats
import matplotlib.pyplot as plt

salarios_a = base.loc[base['grupo'] == 'Grup A', 'salario_eur'].dropna()

salarios_a.plot.hist(bins=20, title='Salarios del grupo A')
plt.xlabel('Salario mensual (€)')
plt.ylabel('Personas')
plt.show()

shapiro = stats.shapiro(salarios_a)
print('Salarios observados:', len(salarios_a))
print(f'Valor p de Shapiro–Wilk: {shapiro.pvalue:.3g}')

En esta encuesta, el histograma tiene una cola hacia los salarios altos. Con 446 salarios observados y un valor p de 3,64 × 10⁻²⁶, rechazamos la hipótesis de normalidad para el grupo A.

## 6. Detectar outliers

Un **outlier** es un valor atípico. Puede ser un dato válido o un error que merece revisión. Seguimos con los salarios del grupo A. El diagrama de caja muestra el tramo central de los salarios, entre el primer y el tercer cuartil. La distancia entre esos cuartiles es el **rango intercuartílico**.

La regla que usamos marca valores a más de 1,5 veces ese rango por debajo o por encima de la caja. Los puntos marcados son casos para revisar. Conservamos sus importes en los análisis de este notebook.

In [ ]:
salarios_a.plot.box(title='Salarios del grupo A')
plt.ylabel('Salario mensual (€)')
plt.show()

q1 = salarios_a.quantile(0.25)
q3 = salarios_a.quantile(0.75)
rango = q3 - q1

por_debajo = salarios_a < q1 - 1.5 * rango
por_encima = salarios_a > q3 + 1.5 * rango
atipicos = salarios_a[por_debajo | por_encima]

print('Salarios atípicos según esta regla:', len(atipicos))
display(atipicos.sort_values(ascending=False).head())

## 7. Analizar variables categóricas

Una variable categórica recoge etiquetas. Aquí miramos los grupos profesionales y las respuestas sobre el coche. Contamos cada categoría y dejamos los vacíos a la vista. Después comparamos los porcentajes de «sí» de A y B sobre quienes respondieron «sí» o «no».

In [ ]:
display(base['grupo'].value_counts(dropna=False))
display(base['tiene_coche'].value_counts(dropna=False))

grupos_ab = base[base['grupo'].isin(['Grup A', 'Grup B'])]
respuesta = grupos_ab['tiene_coche'].map({True: 'si', False: 'no'}).fillna('sin_respuesta')

recuentos = pd.crosstab(grupos_ab['grupo'], respuesta)
recuentos['pct_si'] = 100 * recuentos['si'] / (recuentos['si'] + recuentos['no'])
display(recuentos.round(1))

## 8. Analizar relaciones entre variables

### ¿Se relacionan el salario y los kilómetros? Pearson y Spearman

Tomamos a quienes declararon tener coche y tienen los dos datos. En el gráfico, cada punto es una persona. Mira si los puntos suben, bajan o se dispersan y si hay casos alejados del resto.

- **Pearson** mide la relación lineal, la que se aproxima a una recta.
- **Spearman** compara las posiciones al ordenar cada medida y detecta tendencias que suben o bajan aunque la relación sea curva.

Los dos coeficientes van de −1 a +1. Aquí los usamos para describir los mismos pares de datos. Un valor cercano a cero indica poca relación del tipo que mide el coeficiente. Una asociación observada por sí sola no demuestra una causa.

In [ ]:
con_coche = base[base['tiene_coche'].eq(True)]
pares = con_coche[['salario_eur', 'km_limpios']].dropna()

pares.plot.scatter(x='salario_eur', y='km_limpios', alpha=0.5)
plt.xlabel('Salario mensual (€)')
plt.ylabel('Kilómetros anuales')
plt.show()

pearson = stats.pearsonr(pares['salario_eur'], pares['km_limpios'])
spearman = stats.spearmanr(pares['salario_eur'], pares['km_limpios'])

print('Personas que declararon tener coche:', len(con_coche))
print('Personas con ambos datos:', len(pares))
print(f'Pearson: {pearson[0]:.3f}')
print(f'Spearman: {spearman[0]:.3f}')

De 323 personas que declararon tener coche, 298 tienen los dos datos. Pearson (−0,096) y Spearman (−0,070) muestran una relación negativa débil en esos pares.

### ¿Se asocian el grupo y tener coche? Chi-cuadrado de independencia

Comparamos **Grup A** y **Grup B**. Usamos los recuentos del paso 7. El porcentaje de «sí» se calcula sobre quienes respondieron.

La **hipótesis nula (H₀)** dice que el grupo profesional y la respuesta sobre el coche son independientes. Chi-cuadrado compara los recuentos observados con los que esperaríamos bajo H₀. Usamos las respuestas «sí» y «no» para esa comparación. Miramos cuántas respuestas faltan para entender a qué personas describe el resultado.

El contraste supone observaciones independientes. Como regla práctica, buscamos recuentos observados y esperados de al menos cinco por celda. Con el umbral de 0,05, un valor p menor aporta evidencia de asociación. Un valor igual o mayor indica evidencia insuficiente para rechazar la independencia.

In [ ]:
observados = recuentos[['no', 'si']]
chi = stats.chi2_contingency(observados, correction=False)

esperados = pd.DataFrame(chi.expected_freq, index=observados.index, columns=observados.columns)
display(esperados.round(1))
print(f'Valor p de chi-cuadrado: {chi.pvalue:.3g}')

Declararon tener coche el 50,7 % de quienes respondieron en A y el 51,5 % en B. El valor p de 0,846 indica evidencia insuficiente para rechazar la independencia en esta comparación.

### Ampliación. ¿Cuánto difieren los salarios medios de A y B?

El contraste de **Welch** plantea que los dos grupos tienen la misma media poblacional y admite diferencias de dispersión. Aquí empezamos leyendo la diferencia observada en euros. El contraste supone observaciones independientes. La revisión de normalidad del grupo A describe un grupo y la interpretación de Welch depende de ambos grupos y del diseño de la encuesta.

In [ ]:
salarios_b = base.loc[base['grupo'] == 'Grup B', 'salario_eur'].dropna()
welch = stats.ttest_ind(salarios_a, salarios_b, equal_var=False)

print(f'Diferencia media A menos B: {salarios_a.mean() - salarios_b.mean():.2f} €')
print(f'Valor p de Welch: {welch.pvalue:.3g}')

## 9. Ingeniería y transformación de variables

Podemos crear una columna a partir de los datos que tenemos. Dividimos el kilometraje anual limpio entre doce para obtener un **promedio mensual**. Una ficha con el kilometraje vacío sigue teniendo el promedio vacío.

In [ ]:
base['km_mes_promedio'] = (base['km_limpios'] / 12).round(1)
display(base[['km_limpios', 'km_mes_promedio']].head())

### Añadir contexto de ciudad

En el archivo de contexto, A Coruña aparece escrita como «A Cor\u00f1a» y München como «Munchen». Igualamos esas grafías y corregimos caracteres dañados en la encuesta, como «ValÃ¨ncia». Para detectar el caso València/Valencia, usamos «Valencia» como nombre común en ambas tablas. Así quedan a la vista dos nombres ambiguos, Paris y Valencia. Apartamos sus entradas de la tabla de contexto porque la encuesta solo indica la ciudad.

La unión **izquierda** conserva las fichas de empleados. Donde falta una coincidencia, el índice queda vacío. Conservamos la grafía de la encuesta para leer los resultados.

In [ ]:
grafias = {r'A Cor\u00f1a': 'A Coruña', 'Munchen': 'München', 'València': 'Valencia'}

grafias_encuesta = {
    'A CoruÃ±a': 'A Coruña', 'MÃ¼nchen': 'München', 'ValÃ¨ncia': 'València',
    'MÃ\xa0laga': 'Málaga', 'DÃ¼sseldorf': 'Düsseldorf',
    'KÃ¶ln': 'Köln', 'TromsÃ¸': 'Tromsø',
}

contexto = ciudades.copy()
contexto['clave_ciudad'] = contexto['ciudad'].str.strip().replace(grafias)

ambiguas = contexto.duplicated(subset='clave_ciudad', keep=False)
display(contexto.loc[ambiguas, ['ciudad', 'pais_contexto']])

contexto_unico = contexto.drop_duplicates(subset='clave_ciudad', keep=False)
ciudad_limpia = base['ciudad'].str.strip().replace(grafias_encuesta)
ampliada = base.assign(clave_ciudad=ciudad_limpia.replace(grafias))
ampliada = ampliada.merge(contexto_unico[['clave_ciudad', 'indice_coste_vida']],
                          on='clave_ciudad', how='left')

display(ampliada[['ciudad', 'indice_coste_vida']].head(8))
print('Con índice:', ampliada['indice_coste_vida'].count())
print('Total de personas:', len(ampliada))

Stavanger y Málaga figuran en la encuesta y carecen de entrada en la tabla de contexto. Su índice queda vacío tras la unión. Compara el total de personas con el del paso 3 para ver qué ocurre con las filas.

## 10. Extraer conclusiones

- Empezamos con 1.007 fichas. Tras revisar las repeticiones y los identificadores, analizamos 986 personas. Pudimos añadir el índice de coste de vida a 763.
- Los salarios observados del grupo A tienen una cola hacia los importes altos. Shapiro–Wilk aporta evidencia de que la distribución del grupo se aparta de la normalidad.
- Entre las 298 personas con coche y ambos datos, la relación entre salario y kilometraje es débil. En los grupos A y B, chi-cuadrado aporta evidencia insuficiente para rechazar la independencia entre el grupo y la respuesta sobre el coche.
- La falta de respuestas limita a quiénes describen las comparaciones. Para comunicar un resultado, indica siempre qué filas usaste y cuántas eran.

**Para cerrar.** Elige uno de los resultados y explícalo en dos frases. Incluye una decisión de limpieza que influyó en qué personas entraron en el análisis.

[Guía de contrastes de Machine Learning Mastery](https://machinelearningmastery.com/statistical-hypothesis-tests-in-python-cheat-sheet/) · [Shapiro–Wilk](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.shapiro.html) · [Chi-cuadrado](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html)

---

[Anterior · Spark y HDFS](07_spark_hdfs.ipynb) · [Volver al README](../README.md) · [Siguiente · Visualización de datos](09_data_visualization.ipynb)